# models

> Internal workflow data models for structure decomposition

In [ ]:
#| default_exp core.models

In [ ]:
#| export
from typing import Optional, List, Dict, Any, Protocol, runtime_checkable
from typing_extensions import TypedDict
from dataclasses import dataclass, field, asdict

## Source Record

Standard record format returned by all source providers. This ensures UI components receive a consistent data structure regardless of the underlying source type.

In [ ]:
#| export
class SourceRecord(TypedDict, total=False):
    """Standard record format for source providers."""
    
    # Required fields (used by UI components)
    job_id: str  # Unique record identifier
    plugin_name: str  # Provider/plugin name for display
    text: str  # Content text
    
    # Optional fields
    audio_path: str  # Path to associated audio file
    created_at: str  # Creation timestamp (ISO format or Unix)
    metadata: Dict[str, Any]  # Additional metadata (model, batch_id, etc.)

## Source Provider Protocol

Abstract protocol for content source providers. Each provider represents a queryable source (a database, folder of files, etc.) that can supply `SourceRecord` objects for browsing and `SourceBlock` objects for processing.

In [ ]:
#| export
@runtime_checkable
class SourceProvider(Protocol):
    """Protocol for content source providers."""
    
    @property
    def provider_id(self) -> str:
        """Unique identifier for this provider instance."""
        ...
    
    @property
    def provider_name(self) -> str:
        """Human-readable name for display."""
        ...
    
    @property
    def provider_type(self) -> str:
        """Provider type category (e.g., 'transcription_db', 'local_file')."""
        ...
    
    def query_records(
        self,
        limit: int = 100  # Maximum number of records to return
    ) -> List['SourceRecord']:  # List of source records
        """Query available records from this provider."""
        ...
    
    def get_source_block(
        self,
        record_id: str  # Record identifier (job_id)
    ) -> Optional['SourceBlock']:  # SourceBlock or None if not found
        """Fetch a specific record as a SourceBlock."""
        ...

## Step State Types

TypedDict definitions for workflow step state. These provide type safety and documentation for the state structure used by each workflow phase.

In [ ]:
#| export
class SelectedSource(TypedDict):
    """A selected source in the queue."""
    
    job_id: str  # Unique identifier of the source record
    plugin_name: str  # Provider/plugin name that owns this source

In [ ]:
#| export
class SelectionStepState(TypedDict, total=False):
    """State for Phase 1: Source Selection & Ordering."""
    
    selected_sources: List[SelectedSource]  # Ordered list of selected sources
    grouping_mode: str  # Grouping mode: "audio_path" or "batch_id"
    external_db_paths: List[str]  # List of external database paths
    current_browse_path: str  # Current directory in local files browser
    file_browser_state: Dict[str, Any]  # Serialized BrowserState from file-browser library

In [ ]:
#| export
class DecompositionStepState(TypedDict, total=False):
    """State for Phase 2: Structural Decomposition."""

    # --- Workflow-specific ---
    is_initialized: bool  # Whether segments have been initialized from Phase 1
    segments: List[Dict[str, Any]]  # Working segments (serialized WorkingSegment)
    initial_segments: List[Dict[str, Any]]  # Original segments from initial split (for reset)

    # --- Card stack view state (extractable to cjm-fasthtml-card-stack) ---
    focused_index: int  # Currently focused item index (default 0)
    visible_count: int  # Number of visible cards in viewport
    card_width: int  # Card stack width in rem units
    history: List[List[Dict[str, Any]]]  # Stack of previous item state snapshots

In [ ]:
#| export
class AlignmentStepState(TypedDict, total=False):
    """State for Phase 3: Temporal Alignment."""
    
    vad_chunks: List[Dict[str, Any]]  # VAD chunks (serialized VADChunk)
    focused_chunk_index: Optional[int]  # Currently focused VAD chunk
    focused_segment_index: Optional[int]  # Currently focused segment
    alignment_history: List[Dict[str, Any]]  # History of alignment operations

In [ ]:
#| export
class ReviewStepState(TypedDict, total=False):
    """State for Phase 4: Review & Commit."""
    
    document_title: str  # Final document title
    validation_errors: List[str]  # List of validation issues
    is_validated: bool  # Whether document passed validation

In [ ]:
#| export
class WorkflowStepStates(TypedDict, total=False):
    """Container for all step states in the workflow."""
    
    selection: SelectionStepState  # Phase 1 state
    decomposition: DecompositionStepState  # Phase 2 state
    alignment: AlignmentStepState  # Phase 3 state
    review: ReviewStepState  # Phase 4 state

## Source Block

Represents a raw text block from a transcription source before decomposition.

In [ ]:
#| export
@dataclass
class SourceBlock:
    """A raw text block from a transcription source."""
    
    id: str  # Unique identifier (typically job_id from transcription)
    plugin_name: str  # Source plugin name
    text: str  # Raw transcription text
    media_path: Optional[str] = None  # Path to source media file
    metadata: Dict[str, Any] = field(default_factory=dict)  # Additional metadata from source
    
    def to_dict(self) -> Dict[str, Any]:  # Dictionary representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)

## Working Segment

Represents a segment during the workflow before it's committed to the graph.
This is the mutable working copy used during decomposition and alignment.

In [ ]:
#| export
@dataclass
class WorkingSegment:
    """A segment during workflow processing before graph commit."""
    
    index: int  # Sequence position (0-indexed)
    text: str  # Segment text content
    
    # Source coordinates (from original transcription)
    source_id: Optional[str] = None  # ID of source block
    source_plugin: Optional[str] = None  # Source plugin name
    start_char: Optional[int] = None  # Start character index in source
    end_char: Optional[int] = None  # End character index in source
    
    # Temporal coordinates (from VAD alignment)
    start_time: Optional[float] = None  # Start time in seconds
    end_time: Optional[float] = None  # End time in seconds
    vad_chunk_indices: List[int] = field(default_factory=list)  # Linked VAD chunk indices
    
    def to_dict(self) -> Dict[str, Any]:  # Dictionary representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)
    
    @classmethod
    def from_dict(
        cls,
        data: Dict[str, Any]  # Dictionary representation
    ) -> "WorkingSegment":  # Reconstructed WorkingSegment
        """Create from dictionary."""
        data = data.copy()
        return cls(**data)

## VAD Chunk

Represents a voice activity detection time range from the VAD plugin.

In [ ]:
#| export
@dataclass
class VADChunk:
    """A voice activity detection time range."""
    
    index: int  # Chunk index in sequence
    start_time: float  # Start time in seconds
    end_time: float  # End time in seconds
    assigned_segment: Optional[int] = None  # Index of assigned segment (if any)
    
    @property
    def duration(self) -> float:  # Duration in seconds
        """Calculate chunk duration."""
        return self.end_time - self.start_time
    
    def to_dict(self) -> Dict[str, Any]:  # Dictionary representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)
    
    @classmethod
    def from_dict(
        cls,
        data: Dict[str, Any]  # Dictionary representation
    ) -> "VADChunk":  # Reconstructed VADChunk
        """Create from dictionary."""
        return cls(**data)

## Working Document

Container for all working data during the decomposition workflow.

In [ ]:
#| export
@dataclass
class WorkingDocument:
    """Container for workflow state during structure decomposition."""
    
    title: str = ""  # Document title
    media_type: str = "audio"  # Source media type ('audio', 'video', 'text')
    media_path: Optional[str] = None  # Path to primary source media
    
    # Source blocks (Phase 1 output)
    source_blocks: List[SourceBlock] = field(default_factory=list)  # Ordered source blocks
    
    # Combined raw text (for decomposition)
    combined_text: str = ""  # Concatenated text from all sources
    
    # Working segments (Phase 2 output, Phase 3 input/output)
    segments: List[WorkingSegment] = field(default_factory=list)  # Decomposed segments
    
    # VAD data (Phase 3)
    vad_chunks: List[VADChunk] = field(default_factory=list)  # VAD time ranges
    audio_duration: Optional[float] = None  # Total audio duration in seconds
    
    def to_dict(self) -> Dict[str, Any]:  # Dictionary representation
        """Convert to dictionary for JSON serialization."""
        return {
            'title': self.title,
            'media_type': self.media_type,
            'media_path': self.media_path,
            'source_blocks': [b.to_dict() for b in self.source_blocks],
            'combined_text': self.combined_text,
            'segments': [s.to_dict() for s in self.segments],
            'vad_chunks': [c.to_dict() for c in self.vad_chunks],
            'audio_duration': self.audio_duration
        }
    
    @classmethod
    def from_dict(
        cls,
        data: Dict[str, Any]  # Dictionary representation
    ) -> "WorkingDocument":  # Reconstructed WorkingDocument
        """Create from dictionary."""
        return cls(
            title=data.get('title', ''),
            media_type=data.get('media_type', 'audio'),
            media_path=data.get('media_path'),
            source_blocks=[SourceBlock(**b) for b in data.get('source_blocks', [])],
            combined_text=data.get('combined_text', ''),
            segments=[WorkingSegment.from_dict(s) for s in data.get('segments', [])],
            vad_chunks=[VADChunk.from_dict(c) for c in data.get('vad_chunks', [])],
            audio_duration=data.get('audio_duration')
        )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()